# TinyChirp SincNet-Time TensorFlow

Train a **Conv2D** model (`kernel_size=(k, 1)`) with a SincNet-style learnable frontend on raw audio (time runs along the spatial **height** dimension), export an int8 TFLite model, and write Rust quantized samples in `audio_samples/<model>.rs`.

This mirrors `building_tensorflow/cnn_time.ipynb` but replaces the first convolutional layer with a custom Sinc-like frontend whose learned filters are baked into a static `Conv2D` for inference. Rank-4 tensors throughout keep the graph compatible with microflow on device.

# TinyChirp SincNet-Time TensorFlow

Train a **Conv2D** model (`kernel_size=(k, 1)`) with a SincNet-style learnable frontend on raw audio (time runs along the spatial **height** dimension), export an int8 TFLite model, and write Rust quantized samples in `audio_samples/<model>.rs`.

This mirrors `building_tensorflow/cnn_time.ipynb` but replaces the first convolutional layer with a custom Sinc-like frontend whose learned filters are baked into a static `Conv2D` for inference. Rank-4 tensors throughout keep the graph compatible with microflow on device.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import tensorflow as tf

from utils import (
    TARGET_AUDIO_LEN_TIME,
    SAMPLE_RATE,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
    make_time_datasets,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "sincnet_multi_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
BATCH_SIZE = 32


In [ ]:
def _time_nhwc(audio, label):
    """[B, T, 1] from utils -> [B, T, 1, 1] for Conv2D (time = height)."""
    return tf.expand_dims(audio, -1), label


train_ds, val_ds, test_ds, label_names = make_time_datasets()
train_ds = train_ds.map(_time_nhwc)
val_ds = val_ds.map(_time_nhwc)
test_ds = test_ds.map(_time_nhwc)
num_labels = len(label_names)
print("Classes:", label_names)

## SincNet-style learnable frontend

We define a simplified SincNet-style learnable filterbank as a custom Keras layer that operates directly on the raw waveform. The layer maintains trainable parameters that are passed through a `sin` nonlinearity to produce filters, which are then applied via a 1D convolution.

In [ ]:
from model_parts import SincnetConv
from utils import get_flops_native

In [ ]:
SINCNET_FILTERS = 32
SINCNET_STRIDE = 16
SINCNET_KERNEL_SIZE = 64

CONV_FILTERS = 16
CONV_FILTER_SIZE = 8
CONV_STRIDE = 2

DENSE_HIDDEN = 64


def build_training_model(num_labels: int) -> tf.keras.Model:
    # NHWC: time = height, fixed width 1 — matches microflow rank-4 buffers.
    inputs = tf.keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1, 1))

    x = SincnetConv(
        num_filters=SINCNET_FILTERS,
        kernel_size=SINCNET_KERNEL_SIZE,
        stride=SINCNET_STRIDE,
        sample_rate=SAMPLE_RATE,
        name="sincnet_convolution",
    )(inputs)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(4, 1), name="envelope_pool")(x)

    x = tf.keras.layers.Conv2D(
        filters=CONV_FILTERS,
        kernel_size=(CONV_FILTER_SIZE, 1),
        strides=(CONV_STRIDE, 1),
        padding="same",
        name="temporal_conv_1",
    )(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.AveragePooling2D(pool_size=(4, 1), name="temporal_pool_1")(x)

    x = tf.keras.layers.Conv2D(
        filters=CONV_FILTERS,
        kernel_size=(CONV_FILTER_SIZE, 1),
        strides=(CONV_STRIDE, 1),
        padding="same",
        name="temporal_conv_2",
    )(x)
    x = tf.keras.layers.ReLU()(x)

    # GlobalAveragePooling1D maps to MEAN in TFLite which microflow doesn't support.
    # AveragePooling2D maps to AVERAGE_POOL_2D — pool the full time (height) axis.
    x = tf.keras.layers.AveragePooling2D(
        pool_size=(x.shape[1], 1), padding="valid", name="final_pool"
    )(x)
    x = tf.keras.layers.Flatten(name="flatten")(x)

    x = tf.keras.layers.Dense(DENSE_HIDDEN, activation="relu", name="dense_hidden")(x)
    outputs = tf.keras.layers.Dense(num_labels, activation=None, name="dense_logits")(x)

    return tf.keras.Model(inputs, outputs, name="sincnet_training")

training_model = build_training_model(num_labels)
training_model.summary()

flops = get_flops_native(training_model, batch_size=1)
print(f"Total FLOPs: {flops}")

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

training_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

init_wandb(MODEL_STEM, config={
    "sincnet_num_filters": SINCNET_FILTERS,
    "sincnet_kernel_size": SINCNET_KERNEL_SIZE,
    "sincnet_stride": SINCNET_STRIDE,
    "conv_filters": CONV_FILTERS,
    "conv_filter_size": CONV_FILTER_SIZE,
    "conv_stride": CONV_STRIDE,
    "dense_hidden": DENSE_HIDDEN,
})

history = training_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=44,
    callbacks=get_callbacks(10,5,BATCH_SIZE)
)
finish_wandb()

In [ ]:
from utils import plot_training_history
plot_training_history(history)

## Make frontend filters into a Conv2D inference model


In [ ]:
frontend_layer = training_model.get_layer("sincnet_convolution")
baked_conv_layer = frontend_layer.export_to_conv2d(name="baked_sinc_conv")

infer_inputs = tf.keras.Input(shape=(TARGET_AUDIO_LEN_TIME, 1, 1))
x = baked_conv_layer(infer_inputs)
x = tf.keras.layers.ReLU()(x)
x = training_model.get_layer("envelope_pool")(x)
x = training_model.get_layer("temporal_conv_1")(x)
x = tf.keras.layers.ReLU()(x)
x = training_model.get_layer("temporal_pool_1")(x)
x = training_model.get_layer("temporal_conv_2")(x)
x = tf.keras.layers.ReLU()(x)
x = training_model.get_layer("final_pool")(x)
x = training_model.get_layer("flatten")(x)
x = training_model.get_layer("dense_hidden")(x)
outputs = training_model.get_layer("dense_logits")(x)

inference_model = tf.keras.Model(infer_inputs, outputs, name="sincnet_inference")

for batch_audio, _ in test_ds.take(1):
    logits_train = training_model.predict(batch_audio.numpy(), verbose=0)
    logits_infer = inference_model.predict(batch_audio.numpy(), verbose=0)
    print(f"Max abs diff: {np.max(np.abs(logits_train - logits_infer)):.8f}")

## Export quantized TFLite model and Rust audio samples

We now export an int8-quantized TFLite model using the shared helpers from `building_tensorflow.utils`, and regenerate `audio_sample.rs` clips for the TinyChirp Rust runner.

In [ ]:
rep_batches = build_representative_batches(test_ds, take=100)

try:
    export_keras_model_to_int8_tflite(inference_model, rep_batches, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"TFLite conversion failed: {e}")

In [ ]:
from utils import evaluate_tflite_model

hyperparams = {
    "sincnet_filters": SINCNET_FILTERS,
    "sincnet_kernel_size": SINCNET_KERNEL_SIZE,
    "sincnet_stride": SINCNET_STRIDE,
    "conv_filters": CONV_FILTERS,
    "conv_filter_size": CONV_FILTER_SIZE,
    "conv_stride": CONV_STRIDE,
    "dense_hidden": DENSE_HIDDEN,
    "target_audio_len": TARGET_AUDIO_LEN_TIME,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")